# LeetCode #189 — Rotate Array: xoay tại chỗ bằng ba lần đảo

**Mức độ:** Medium · **Chủ đề:** Array, Math, Two Pointers

Skeleton trên LeetCode:

```python
class Solution:
    def rotate(self, nums: List[int], k: int) -> None:
        """
        Do not return anything, modify nums in-place instead.
        """
```

Mục tiêu của notebook:

1. Hiểu xoay phải `k` bước: phần tử ở vị trí `i` đi tới vị trí `(i + k) % n`.
2. Luôn **thu gọn** `k %= n` (xoay `n` bước là quay về chỗ cũ).
3. Ba cách: mảng phụ `O(n)`, **ba lần đảo** `O(1)`, **thay thế theo chu trình** (dùng `gcd`) `O(1)`.
4. Bẫy Python: `nums = …` không sửa mảng gốc; phải `nums[:] = …` hoặc đổi chỗ tại chỗ.

# 1. Bài toán là gì?

Cho mảng `nums`, xoay mảng sang **phải** `k` bước (`k` không âm). Sửa **tại chỗ**.

**Constraints (theo đề):**

- `1 <= nums.length <= 10^5`
- `-2^31 <= nums[i] <= 2^31 - 1`
- `0 <= k <= 10^5`

**Follow up (theo đề):** nghĩ ra ít nhất **ba** cách; làm được tại chỗ với `O(1)` bộ nhớ phụ không?

---

## Ví dụ

```text
nums = [1,2,3,4,5,6,7], k = 3   ->  [5,6,7,1,2,3,4]
nums = [-1,-100,3,99],  k = 2   ->  [3,99,-1,-100]
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Hai con trỏ (two pointers)
2. Số học cơ bản — số nguyên tố, sàng, ƯCLN, lũy thừa của p trong n!

Cuối mục có phần **2.3 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Hai con trỏ (two pointers)

### Ý tưởng

Dùng **hai chỉ số** di chuyển trên mảng/chuỗi theo quy tắc, để thay vòng lặp lồng `O(n²)` bằng một lượt `O(n)`. Hai kiểu chính:

**1. Hai đầu tiến vào nhau** (thường trên mảng đã sắp xếp):

```text
[1, 2, 4, 7, 11, 15], tìm hai số có tổng 15
 L                 R     1 + 15 = 16 > 15 → R lùi (cần nhỏ hơn)
 L             R         1 + 11 = 12 < 15 → L tiến (cần lớn hơn)
    L          R         2 + 11 = 13 < 15 → L tiến
       L       R         4 + 11 = 15 ✓
```

Vì sao được phép bỏ? Khi `a[L] + a[R] > target`, mọi cặp `(L', R)` với `L' > L` còn lớn hơn → `R` không thể là đáp án, bỏ `R`.

**2. Cùng chiều — con trỏ đọc và con trỏ ghi** (sửa mảng tại chỗ):

```text
xóa mọi số 0, giữ thứ tự: [0, 1, 0, 3, 12]
read đi qua từng phần tử; gặp số khác 0 thì chép vào vị trí write, write += 1
→ [1, 3, 12, ...]
```

In [1]:
def tong_hai_so_da_sap_xep(a, target):
    L, R = 0, len(a) - 1
    while L < R:
        s = a[L] + a[R]
        print(f"   L={L} R={R}: {a[L]} + {a[R]} = {s}")
        if s == target:
            return L, R
        if s < target:
            L += 1
        else:
            R -= 1
    return None

print("→", tong_hai_so_da_sap_xep([1, 2, 4, 7, 11, 15], 15))

def bo_so_khong(a):
    write = 0
    for read in range(len(a)):
        if a[read] != 0:
            a[write] = a[read]
            write += 1
    return a[:write]

print("bỏ số 0:", bo_so_khong([0, 1, 0, 3, 12]))

def la_palindrome(s):
    i, j = 0, len(s) - 1
    while i < j:
        if s[i] != s[j]:
            return False
        i += 1; j -= 1
    return True
print("'racecar' palindrome?", la_palindrome("racecar"), "| 'abca'?", la_palindrome("abca"))

   L=0 R=5: 1 + 15 = 16
   L=0 R=4: 1 + 11 = 12
   L=1 R=4: 2 + 11 = 13
   L=2 R=4: 4 + 11 = 15
→ (2, 4)
bỏ số 0: [1, 3, 12]
'racecar' palindrome? True | 'abca'? False


### Đảo ngược một đoạn tại chỗ

Hai con trỏ ở hai đầu, đổi chỗ rồi tiến vào trong — dùng rất nhiều (đảo mảng, xoay mảng, đảo từ).

In [2]:
def dao_doan(a, i, j):
    while i < j:
        a[i], a[j] = a[j], a[i]
        i += 1; j -= 1

arr_tp = [1, 2, 3, 4, 5, 6]
dao_doan(arr_tp, 1, 4)
print("đảo đoạn [1..4]:", arr_tp)

đảo đoạn [1..4]: [1, 5, 4, 3, 2, 6]


### Lỗi hay gặp

- Quên **sắp xếp** trước khi dùng hai đầu cho bài tổng.
- Điều kiện `L <= R` thay vì `L < R` → dùng một phần tử hai lần.
- Không chứng minh được vì sao bỏ một đầu là an toàn → dễ bỏ mất đáp án.
- Con trỏ ghi vượt con trỏ đọc (không thể xảy ra nếu `write` chỉ tăng khi ghi).

## 2.2 Số học cơ bản — số nguyên tố, sàng, ƯCLN, lũy thừa của p trong n!

### Số nguyên tố và kiểm tra bằng thử chia

`p >= 2` là **số nguyên tố** nếu chỉ chia hết cho 1 và chính nó. Để kiểm tra `n`, chỉ cần thử các ước `d` với `d² <= n`: nếu `n = a · b` với `a <= b` thì `a <= √n`.

In [3]:
from math import isqrt, gcd

def is_prime_demo(n):
    if n < 2:
        return False
    for d in range(2, isqrt(n) + 1):
        if n % d == 0:
            return False
    return True

print([x for x in range(30) if is_prime_demo(x)])
print("isqrt(10**12 + 1) =", isqrt(10**12 + 1), "(căn nguyên chính xác, không lỗi làm tròn số thực)")

[2, 3, 5, 7, 11, 13, 17, 19, 23, 29]
isqrt(10**12 + 1) = 1000000 (căn nguyên chính xác, không lỗi làm tròn số thực)


### Sàng Eratosthenes: mọi số nguyên tố `< n` trong O(n log log n)

Ý tưởng: đi từ 2 lên; gặp số `p` chưa bị gạch thì `p` nguyên tố, **gạch mọi bội** của nó. Bắt đầu gạch từ `p²` (các bội nhỏ hơn đã bị gạch bởi ước nhỏ hơn), và chỉ cần `p² < n`.

```text
2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19
    x   x   x x  x     x     x  x  x     x       (gạch bội của 2 từ 4, của 3 từ 9)
```

In [4]:
def sieve_demo(n):
    if n < 3:
        return []
    is_p = bytearray([1]) * n
    is_p[0] = is_p[1] = 0
    for p in range(2, isqrt(n - 1) + 1):
        if is_p[p]:
            is_p[p * p::p] = bytes(len(range(p * p, n, p)))
    return [i for i in range(n) if is_p[i]]

print(sieve_demo(30), len(sieve_demo(10**5)))

[2, 3, 5, 7, 11, 13, 17, 19, 23, 29] 9592


### Ước chung lớn nhất: thuật toán Euclid

`gcd(a, b) = gcd(b, a mod b)`, dừng khi `b = 0`. Số bước `O(log min(a, b))`. Python có sẵn `math.gcd`.

In [5]:
def gcd_demo(a, b):
    steps = []
    while b:
        steps.append((a, b))
        a, b = b, a % b
    return a, steps

g, st = gcd_demo(252, 105)
print("gcd(252, 105) =", g, "| các bước:", st, "| math.gcd:", gcd(252, 105))

gcd(252, 105) = 21 | các bước: [(252, 105), (105, 42), (42, 21)] | math.gcd: 21


Ứng dụng: các bước nhảy `+k (mod n)` xuất phát từ 0 đi qua đúng `n / gcd(n, k)` vị trí rồi quay lại 0 — chia các vị trí thành `gcd(n, k)` vòng.

In [6]:
n_demo, k_demo = 6, 4
start, cycle = 0, [0]
while (cycle[-1] + k_demo) % n_demo != start:
    cycle.append((cycle[-1] + k_demo) % n_demo)
print(f"n={n_demo}, k={k_demo}: vòng từ 0 = {cycle}, độ dài {len(cycle)} = n / gcd = {n_demo // gcd(n_demo, k_demo)}")

n=6, k=4: vòng từ 0 = [0, 4, 2], độ dài 3 = n / gcd = 3


### Lũy thừa của số nguyên tố p trong n! (công thức Legendre)

Số lần `p` xuất hiện trong phân tích `n!` là `⌊n/p⌋ + ⌊n/p²⌋ + ⌊n/p³⌋ + …`: `⌊n/p⌋` số chia hết cho `p` mỗi số góp 1, `⌊n/p²⌋` số chia hết cho `p²` góp thêm 1, …

In [7]:
from math import factorial

def legendre_demo(n, p):
    total, q = 0, p
    while q <= n:
        total += n // q
        q *= p
    return total

def count_p_brute(n, p):
    f, c = factorial(n), 0
    while f % p == 0:
        f //= p; c += 1
    return c

for n_, p_ in [(10, 2), (25, 5), (100, 3)]:
    print(f"lũy thừa của {p_} trong {n_}! = {legendre_demo(n_, p_)} (đếm trực tiếp: {count_p_brute(n_, p_)})")

lũy thừa của 2 trong 10! = 8 (đếm trực tiếp: 8)
lũy thừa của 5 trong 25! = 6 (đếm trực tiếp: 6)
lũy thừa của 3 trong 100! = 48 (đếm trực tiếp: 48)


### Lỗi hay gặp

- Kiểm tra nguyên tố tới `n` thay vì `√n` → chậm; dùng `int(n ** 0.5)` có thể lệch với số lớn → dùng `isqrt`.
- Sàng gạch từ `2p` thay vì `p²` (vẫn đúng nhưng chậm hơn); quên 0 và 1 không nguyên tố.
- Tính `n!` thật rồi đếm → số khổng lồ, rất chậm.

## 2.3 Áp dụng vào bài này

Xoay phải `k` bước tại chỗ bằng **ba lần đảo** (đảo cả mảng, đảo `k` phần tử đầu, đảo phần còn lại) — mỗi lần đảo dùng **hai con trỏ**. Cách thay thế là **dịch vòng**: đặt mỗi phần tử vào vị trí `(i + k) % n`; các bước nhảy chia mảng thành đúng **`gcd(n, k)` vòng** — kiến thức ƯCLN ở trên giải thích vì sao phải lặp lại từ nhiều điểm xuất phát. Nhớ `k %= n`.

# 3. Nền tảng

In [8]:
from typing import List
from math import gcd
import random, time

nums = [1, 2, 3, 4, 5, 6, 7]
n, k = len(nums), 3
print("vị trí mới của phần tử ở i:", {i: (i + k) % n for i in range(n)})
print("xoay 10 bước = xoay", 10 % n, "bước")

vị trí mới của phần tử ở i: {0: 3, 1: 4, 2: 5, 3: 6, 4: 0, 5: 1, 6: 2}
xoay 10 bước = xoay 3 bước


Phần tử cuối "quấn vòng" sang đầu → xoay phải `k` bước nghĩa là `k` phần tử **cuối** lên **đầu**, giữ thứ tự.

## Bẫy: gán lại tên biến không sửa mảng gốc

In [9]:
def wrong_rebind(nums, k):
    nums = nums[-k:] + nums[:-k]          # chỉ đổi biến cục bộ

def right_slice_assign(nums, k):
    nums[:] = nums[-k:] + nums[:-k]       # sửa NỘI DUNG của list gốc

a = [1, 2, 3, 4]; wrong_rebind(a, 1); print("gán lại tên :", a)
b = [1, 2, 3, 4]; right_slice_assign(b, 1); print("nums[:] = …:", b)

gán lại tên : [1, 2, 3, 4]
nums[:] = …: [4, 1, 2, 3]


# 4. Cách 1: mảng phụ (hoặc cắt lát)

Đặt `out[(i + k) % n] = nums[i]` rồi chép lại. `O(n)` thời gian, `O(n)` bộ nhớ.

In [10]:
def rotate_extra(nums: List[int], k: int) -> None:
    n = len(nums)
    out = [0] * n
    for i in range(n):
        out[(i + k) % n] = nums[i]
    nums[:] = out

a = [1, 2, 3, 4, 5, 6, 7]; rotate_extra(a, 3); print(a)

[5, 6, 7, 1, 2, 3, 4]


Bản cắt lát `nums[:] = nums[-k:] + nums[:-k]` gọn hơn nhưng cũng tạo list tạm `O(n)` — và **sai khi `k % n == 0`** nếu quên thu gọn (xem mục 10).

# 5. Cách 2 (chính): ba lần đảo

Viết mảng là `A B` với `B` là `k` phần tử cuối. Muốn `B A`.

```text
đảo cả mảng:        rev(B) rev(A)        (rev(XY) = rev(Y) rev(X))
đảo k phần tử đầu:  B      rev(A)
đảo phần còn lại:   B      A             ✓
```

```text
[1,2,3,4,5,6,7], k = 3
đảo cả:        [7,6,5,4,3,2,1]
đảo [0..2]:    [5,6,7,4,3,2,1]
đảo [3..6]:    [5,6,7,1,2,3,4]
```

In [11]:
def reverse(nums: List[int], i: int, j: int) -> None:
    while i < j:
        nums[i], nums[j] = nums[j], nums[i]
        i += 1
        j -= 1

def rotate_reverse(nums: List[int], k: int) -> None:
    n = len(nums)
    k %= n
    reverse(nums, 0, n - 1)
    reverse(nums, 0, k - 1)
    reverse(nums, k, n - 1)

a = [1, 2, 3, 4, 5, 6, 7]; rotate_reverse(a, 3); print(a)

[5, 6, 7, 1, 2, 3, 4]


# 6. Cách 3: thay thế theo chu trình

Đặt phần tử ở `i` vào `(i + k) % n`; phần tử bị chiếm chỗ lại đi tới `(i + 2k) % n`, … cho tới khi quay về `i` — đó là một **chu trình**. Có đúng `g = gcd(n, k)` chu trình, mỗi chu trình dài `n / g`. Duyệt từ `start = 0, 1, …, g − 1`.

```text
n = 6, k = 2, gcd = 2:  chu trình 0 → 2 → 4 → 0   và   1 → 3 → 5 → 1
```

In [12]:
def rotate_cycles(nums: List[int], k: int) -> None:
    n = len(nums)
    k %= n
    if k == 0:
        return
    for start in range(gcd(n, k)):
        cur, prev = start, nums[start]
        while True:
            nxt = (cur + k) % n
            nums[nxt], prev = prev, nums[nxt]     # đặt prev vào nxt, cầm phần tử bị chiếm chỗ
            cur = nxt
            if cur == start:
                break

a = [1, 2, 3, 4, 5, 6]; rotate_cycles(a, 2); print(a)
a = [1, 2, 3, 4, 5, 6, 7]; rotate_cycles(a, 3); print(a)

[5, 6, 1, 2, 3, 4]


[5, 6, 7, 1, 2, 3, 4]


# 7. Mô phỏng

In [13]:
def trace_reverse(nums, k):
    a = list(nums)
    n = len(a); k %= n
    print(f"ban đầu (k = {k}): {a}")
    reverse(a, 0, n - 1);  print(f"đảo cả        : {a}")
    reverse(a, 0, k - 1);  print(f"đảo [0..{k - 1}]    : {a}")
    reverse(a, k, n - 1);  print(f"đảo [{k}..{n - 1}]    : {a}")

def trace_cycles(nums, k):
    a = list(nums); n = len(a); k %= n
    print(f"n = {n}, k = {k}, gcd = {gcd(n, k)} chu trình")
    for start in range(gcd(n, k)):
        path, cur = [start], start
        while True:
            cur = (cur + k) % n
            path.append(cur)
            if cur == start:
                break
        print("   chu trình vị trí:", " → ".join(map(str, path)))

trace_reverse([1, 2, 3, 4, 5, 6, 7], 3)
trace_cycles([1, 2, 3, 4, 5, 6], 2)
trace_cycles([1, 2, 3, 4, 5, 6, 7], 3)

ban đầu (k = 3): [1, 2, 3, 4, 5, 6, 7]


đảo cả        : [7, 6, 5, 4, 3, 2, 1]
đảo [0..2]    : [5, 6, 7, 4, 3, 2, 1]
đảo [3..6]    : [5, 6, 7, 1, 2, 3, 4]
n = 6, k = 2, gcd = 2 chu trình
   chu trình vị trí: 0 → 2 → 4 → 0
   chu trình vị trí: 1 → 3 → 5 → 1
n = 7, k = 3, gcd = 1 chu trình
   chu trình vị trí: 0 → 3 → 6 → 2 → 5 → 1 → 4 → 0


# 8. Phiên bản đúng format LeetCode

In [14]:
class Solution:
    def rotate(self, nums: List[int], k: int) -> None:
        n = len(nums)
        k %= n

        def rev(i: int, j: int) -> None:
            while i < j:
                nums[i], nums[j] = nums[j], nums[i]
                i += 1
                j -= 1

        rev(0, n - 1)
        rev(0, k - 1)
        rev(k, n - 1)

In [15]:
def check(name, nums, k, expected):
    a = list(nums)
    out = Solution().rotate(a, k)
    assert out is None and a == expected, (name, a, expected)
    print(f"{name:<14} PASS  {a}")


check("Ví dụ 1", [1, 2, 3, 4, 5, 6, 7], 3, [5, 6, 7, 1, 2, 3, 4])
check("Ví dụ 2", [-1, -100, 3, 99], 2, [3, 99, -1, -100])
check("k = 0", [1, 2, 3], 0, [1, 2, 3])
check("k = n", [1, 2, 3], 3, [1, 2, 3])
check("k > n", [1, 2], 5, [2, 1])
check("Một phần tử", [9], 7, [9])

random.seed(189)
for _ in range(3000):
    n = random.randint(1, 12)
    nums = [random.randint(-9, 9) for _ in range(n)]
    k = random.randint(0, 30)
    exp = [nums[(i - k) % n] for i in range(n)]
    for f in (Solution().rotate, rotate_extra, rotate_cycles):
        a = list(nums); f(a, k)
        assert a == exp, (f, nums, k)
print("3000 bộ ngẫu nhiên: ba cách đều đúng")

big = list(range(10**5))
for name, f in [("ba lần đảo", Solution().rotate), ("chu trình", rotate_cycles), ("mảng phụ", rotate_extra)]:
    a = list(big); t0 = time.time(); f(a, 12345); t1 = time.time()
    print(f"n = 10^5, {name:<10}: {t1 - t0:.3f}s")

Ví dụ 1        PASS  [5, 6, 7, 1, 2, 3, 4]
Ví dụ 2        PASS  [3, 99, -1, -100]
k = 0          PASS  [1, 2, 3]
k = n          PASS  [1, 2, 3]
k > n          PASS  [2, 1]
Một phần tử    PASS  [9]


3000 bộ ngẫu nhiên: ba cách đều đúng


n = 10^5, ba lần đảo: 0.020s


n = 10^5, chu trình : 0.044s


n = 10^5, mảng phụ  : 0.016s


# 9. Vì sao đúng? Độ phức tạp?

**Ba lần đảo:** với `A` dài `n − k`, `B` dài `k`: `rev(AB) = rev(B) rev(A)`; đảo riêng khối đầu (dài `k`) và khối sau → `B A`, đúng là xoay phải `k`.

**Chu trình:** ánh xạ `i ↦ (i + k) mod n` chia `{0, …, n−1}` thành các chu trình; mỗi chu trình gồm các `i` cùng số dư khi chia cho `g = gcd(n, k)`, có độ dài `n/g`. Đi hết một chu trình đặt đúng mọi phần tử của nó; `g` chu trình bắt đầu từ `0..g−1` phủ hết mảng.

| Cách | Thời gian | Bộ nhớ phụ |
|---|---|---|
| Mảng phụ / cắt lát | `O(n)` | `O(n)` |
| Ba lần đảo | `O(n)` (mỗi phần tử đổi chỗ ≤ 2 lần) | `O(1)` |
| Chu trình | `O(n)` (mỗi phần tử di chuyển 1 lần) | `O(1)` |

# 10. Những lỗi dễ mắc

## Lỗi 1: Quên `k %= n`

In [16]:
def wrong_no_mod(nums, k):
    n = len(nums)
    reverse(nums, 0, n - 1)
    reverse(nums, 0, k - 1)                        # SAI khi k > n: đoạn [0..k-1] vượt khỏi mảng
    reverse(nums, k, n - 1)

a = [1, 2, 3]
try:
    wrong_no_mod(a, 4)
    print("k = 4:", a)
except IndexError as e:
    print("k = 4 → IndexError:", e)

k = 4 → IndexError: list index out of range


Với `k = 4 > n = 3`, đoạn `[0..3]` vượt mảng → `IndexError` (còn `k = n` thì tình cờ đúng: đảo cả mảng hai lần là giữ nguyên). Đề cho `k` tới `10^5` trong khi `n` có thể chỉ là 1 — luôn thu gọn `k %= n` trước.

## Lỗi 2: Cắt lát khi `k % n == 0` mà không thu gọn `k`

In [17]:
def wrong_slice(nums, k):
    nums[:] = nums[-k:] + nums[:-k]

for k in [0, 3]:
    a = [1, 2, 3]
    try:
        wrong_slice(a, k)
        print(f"k = {k}: {a}")
    except Exception as e:
        print(f"k = {k}: {type(e).__name__}")

k = 0: [1, 2, 3]
k = 3: [1, 2, 3]


`k = 0`: `nums[-0:]` = `nums[0:]` = **cả mảng**, `nums[:-0]` = `nums[:0]` = rỗng → tình cờ đúng. Nhưng với `k = 5`, `nums[-5:]` lấy cả mảng (Python không báo lỗi khi cắt vượt) và `nums[:-5]` rỗng → **không** xoay gì cả trong khi đáng lẽ xoay `5 % 3 = 2` bước:

In [18]:
a = [1, 2, 3]; wrong_slice(a, 5); print("k = 5, sai:", a, "| đúng:", [2, 3, 1])

k = 5, sai: [1, 2, 3] | đúng: [2, 3, 1]


## Lỗi 3: Chu trình chỉ chạy một lần từ `start = 0`

In [19]:
def wrong_one_cycle(nums, k):
    n = len(nums); k %= n
    cur, prev = 0, nums[0]
    while True:
        nxt = (cur + k) % n
        nums[nxt], prev = prev, nums[nxt]
        cur = nxt
        if cur == 0:
            break

a = [1, 2, 3, 4, 5, 6]; wrong_one_cycle(a, 2); print("Sai :", a, "| Đúng:", [5, 6, 1, 2, 3, 4])

Sai : [5, 2, 1, 4, 3, 6] | Đúng: [5, 6, 1, 2, 3, 4]


Khi `gcd(n, k) > 1` có nhiều chu trình; chỉ xử lý chu trình chứa vị trí 0 thì các vị trí lẻ không được di chuyển.

## Lỗi 4: Xoay từng bước một, `k` lần

`k` lần "lấy phần tử cuối chèn lên đầu" → `O(n · k)`, với `n = k = 10^5` là `10^10` thao tác.

# 11. Mẹo nhận diện pattern

- **Xoay / hoán đổi hai khối kề nhau tại chỗ** → ba lần đảo.
- **Hoán vị theo công thức chỉ số** (`i ↦ (i + k) mod n`) → đi theo chu trình; số chu trình = `gcd`.
- **Chỉ số vòng** → luôn thu gọn bằng `% n`.

Bài liên quan: **#61 Rotate List**, **#186 Reverse Words in a String II**, **#796 Rotate String** (Easy), **#48 Rotate Image**, **#1752 Check if Array Is Sorted and Rotated** (Easy).

# 12. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Xoay `[10, 20, 30, 40, 50, 60]` với `k = 4` bằng ba lần đảo; liệt kê các chu trình khi dùng cách thay thế.

## Bài 2 — Điền chỗ trống (ba lần đảo)

In [20]:
def exercise_2_rotate(nums: List[int], k: int) -> None:
    n = len(nums)
    k %= n
    # TODO: ba lần reverse(...)
    pass


# e2 = [1, 2, 3, 4, 5, 6, 7]; exercise_2_rotate(e2, 3); assert e2 == [5, 6, 7, 1, 2, 3, 4]
# e2b = [-1, -100, 3, 99]; exercise_2_rotate(e2b, 2); assert e2b == [3, 99, -1, -100]
# e2c = [1, 2]; exercise_2_rotate(e2c, 4); assert e2c == [1, 2]

## Bài 3 — Biến thể: xoay **trái** `k` bước

In [21]:
def exercise_3_rotate_left(nums: List[int], k: int) -> None:
    # TODO
    pass


# e3 = [1, 2, 3, 4, 5]; exercise_3_rotate_left(e3, 2); assert e3 == [3, 4, 5, 1, 2]
# e3b = [1, 2, 3]; exercise_3_rotate_left(e3b, 7); assert e3b == [2, 3, 1]

## Bài 4 — Biến thể: xoay phải chỉ đoạn `nums[l..r]`

In [22]:
def exercise_4_rotate_range(nums: List[int], l: int, r: int, k: int) -> None:
    # TODO
    pass


# e4 = [1, 2, 3, 4, 5, 6]; exercise_4_rotate_range(e4, 1, 4, 1); assert e4 == [1, 5, 2, 3, 4, 6]
# e4b = [1, 2, 3]; exercise_4_rotate_range(e4b, 0, 2, 3); assert e4b == [1, 2, 3]

## Bài 5 — Mini challenge: đổi chỗ hai khối kề nhau **khác độ dài**

`nums = A + B` với `A` dài `p`. Biến thành `B + A` tại chỗ. (Đây chính là xoay trái `p` bước — hãy viết lại bằng cách **chu trình** để mỗi phần tử chỉ di chuyển một lần.)

In [23]:
def exercise_5_swap_blocks(nums: List[int], p: int) -> None:
    # TODO
    pass


# e5 = [1, 2, 9, 8, 7]; exercise_5_swap_blocks(e5, 2); assert e5 == [9, 8, 7, 1, 2]
# e5b = [1, 2, 3, 4]; exercise_5_swap_blocks(e5b, 0); assert e5b == [1, 2, 3, 4]

# 13. Gợi ý

**Bài 1**
- Mức 1: chạy `trace_reverse` và `trace_cycles` sau khi tự làm.

**Bài 3**
- Mức 1: xoay trái `k` = xoay phải `n − k % n`. Hoặc: đảo `[0..k−1]`, đảo `[k..n−1]`, rồi đảo cả.

**Bài 4**
- Mức 1: `m = r − l + 1`, `k %= m`; ba lần đảo trên `[l..r]`, `[l..l+k−1]`, `[l+k..r]`.

**Bài 5**
- Mức 1: đổi chỗ khối = xoay **phải** `n − p` bước; dùng `rotate_cycles(nums, (n − p) % n)`.

# 14. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [24]:
trace_reverse([10, 20, 30, 40, 50, 60], 4)
trace_cycles([10, 20, 30, 40, 50, 60], 4)

ban đầu (k = 4): [10, 20, 30, 40, 50, 60]
đảo cả        : [60, 50, 40, 30, 20, 10]
đảo [0..3]    : [30, 40, 50, 60, 20, 10]
đảo [4..5]    : [30, 40, 50, 60, 10, 20]
n = 6, k = 4, gcd = 2 chu trình
   chu trình vị trí: 0 → 4 → 2 → 0
   chu trình vị trí: 1 → 5 → 3 → 1


## Đáp án Bài 2

In [25]:
def answer_2_rotate(nums: List[int], k: int) -> None:
    rotate_reverse(nums, k)

e2 = [1, 2, 3, 4, 5, 6, 7]; answer_2_rotate(e2, 3); assert e2 == [5, 6, 7, 1, 2, 3, 4]
e2b = [-1, -100, 3, 99]; answer_2_rotate(e2b, 2); assert e2b == [3, 99, -1, -100]
e2c = [1, 2]; answer_2_rotate(e2c, 4); assert e2c == [1, 2]
print("Bài 2 PASS (giống rotate_reverse ở mục 5)")

Bài 2 PASS (giống rotate_reverse ở mục 5)


## Đáp án Bài 3

In [26]:
def answer_3_rotate_left(nums: List[int], k: int) -> None:
    n = len(nums)
    k %= n
    reverse(nums, 0, k - 1)
    reverse(nums, k, n - 1)
    reverse(nums, 0, n - 1)

e3 = [1, 2, 3, 4, 5]; answer_3_rotate_left(e3, 2); assert e3 == [3, 4, 5, 1, 2]
e3b = [1, 2, 3]; answer_3_rotate_left(e3b, 7); assert e3b == [2, 3, 1]
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [27]:
def answer_4_rotate_range(nums: List[int], l: int, r: int, k: int) -> None:
    m = r - l + 1
    k %= m
    reverse(nums, l, r)
    reverse(nums, l, l + k - 1)
    reverse(nums, l + k, r)

e4 = [1, 2, 3, 4, 5, 6]; answer_4_rotate_range(e4, 1, 4, 1); assert e4 == [1, 5, 2, 3, 4, 6]
e4b = [1, 2, 3]; answer_4_rotate_range(e4b, 0, 2, 3); assert e4b == [1, 2, 3]
for _ in range(2000):
    n = random.randint(1, 10); a = list(range(n))
    l = random.randint(0, n - 1); r = random.randint(l, n - 1); k = random.randint(0, 12)
    seg = a[l:r + 1]; kk = k % len(seg)
    exp = a[:l] + (seg[-kk:] + seg[:-kk] if kk else seg) + a[r + 1:]
    answer_4_rotate_range(a, l, r, k)
    assert a == exp
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [28]:
def answer_5_swap_blocks(nums: List[int], p: int) -> None:
    n = len(nums)
    if n:
        rotate_cycles(nums, (n - p) % n)

e5 = [1, 2, 9, 8, 7]; answer_5_swap_blocks(e5, 2); assert e5 == [9, 8, 7, 1, 2]
e5b = [1, 2, 3, 4]; answer_5_swap_blocks(e5b, 0); assert e5b == [1, 2, 3, 4]
for _ in range(2000):
    n = random.randint(1, 10); a = list(range(n)); p = random.randint(0, n)
    exp = a[p:] + a[:p]
    answer_5_swap_blocks(a, p)
    assert a == exp
print("Bài 5 PASS")

Bài 5 PASS


# 15. Tổng kết

> `k %= n`, rồi **đảo cả mảng → đảo `k` phần tử đầu → đảo phần còn lại**: `O(n)` thời gian, `O(1)` bộ nhớ. Cách khác `O(1)`: đi theo `gcd(n, k)` chu trình `i → (i + k) % n`. Trong Python nhớ sửa **nội dung** (`nums[:] = …` hoặc đổi chỗ), không gán lại tên.

---

## Checklist trước khi nộp

- [ ] `k %= n` trước mọi thứ?
- [ ] Sửa tại chỗ (không `nums = …`)?
- [ ] Ba đoạn đảo đúng: `[0..n−1]`, `[0..k−1]`, `[k..n−1]`?
- [ ] Cách chu trình: lặp đủ `gcd(n, k)` chu trình?